# Machine Learning Homework - Group 22
## Notebook 4 - Model & Assessment under a chronological protocol

**Objective**: Choose an assessment strategy and suitable classification metrics, explain why they fit the problem, train at least one
model using `train.csv` and generate predictions for every observation in `test.csv`

**Inputs**: `data/train_clean.csv`, `data/test_clean.csv` (Notebook 2), `data/selected_columns.csv` (Notebook 3), `data/train_raw.csv` (only to recover `purchase_date` for a chronological check), `data/sample_submission.csv` (the submission structure)

**Outputs**: `deliverables/ML_Group_22_Version05.csv`, `artifacts/metrics/*` (assessment, protocol comparison, scaling check, model comparison, threshold curves, holdout by customer type), `artifacts/figures/nb04_assessment.png`

### What the earlier notebooks force us to do here

| Earlier finding | Consequence for modelling |
|---|---|
|NB01: the same `customer_id` appears in many rows|A random split can put the same customer in train and validation and inflate the score, so `GroupKFold` on `customer_id` is measured as the never-seen-customer floor|
| NB01: `train` happens in 2015-2025, `test` in 2025-2026, and `test` customers are longer-tenured | Every row we are scored on is later than every row we can fit on, so model selection uses `TimeSeriesSplit`; a random or grouped split fits on the future to score the past|
| NB01: classes are close to balanced (44/56)| We tune `class_weight`, as adjusting it can shift the decision boundary and improve the F1 score|
| NB02: sentinels and structural nulls became NaN + flags | Those flags are ordinary candidate features now, nothing is re-preprocessed here |
|NB03: the selected set of columns, chosen by measured cross-validation F1 on chronological folds | The selected set is used and the full set is scored alongside it so the selection stays verifiable |
| NB02: only the numeric columns were standardised, the binary flags and dummies were left at 0/1| Re-standardising the whole matrix inside every fold divides a rare dummy by a standard deviation of about 0.07 and hands it an unearned weight. Cell 5 will measure both options|

### Models considered
`DummyClassifier` baseline, `LogisticRegression` (`C` and `class_weight`), `DecisionTreeClassifier` (`max_depth`, `min_samples_leaf`) and
`PCA` + `LogisticRegression`.

### Order of operations
1. Environment
2. Load and verify
3. Candidate feature sets
4. Validation protocol
5. Scaling check and model selection
6. Classification threshold optimization
7. Final model
8. Assessment
9. `Test` prediction
10. Export results and final summary

### Cell 1 - Environment:
Setup libraries, theme, environment and variables used for modelling.

In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import math
from sklearn.base import clone
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import (StratifiedKFold, GroupKFold, TimeSeriesSplit, GridSearchCV, cross_validate)
from sklearn.metrics import (f1_score, accuracy_score, roc_auc_score, confusion_matrix, classification_report, precision_recall_curve, roc_curve, get_scorer)
pd.set_option('display.max_columns', None)

ROOT = Path.cwd().parent
DATA_DIR = ROOT / "data"
FIG_DIR = ROOT / "artifacts" / "figures"
MET_DIR = ROOT / "artifacts" / "metrics"
DELIV_DIR = ROOT / "deliverables"

TARGET = "repeat_purchase_90d"
RANDOM_STATE = 42

SUBMISSION_FILE = "ML_Group_22_Version05.csv"
PLOT_BLUE = "#1779A8"
PLOT_ORANGE = "#C96A00"
ID_COLS = ["ID", "customer_id"]

def guide(ax, text):
    ax.set_title(ax.get_title() + "\n" + text, fontsize=8)

sns.set_theme(style="whitegrid")
print("Environment ready.")
print("  submission goes to:", DELIV_DIR)

### Cell 2 - Load and verify:
Read created matrices from Notebook 2, feature selection from Notebook 3, raw dates needed for a chronological check and verifies with some `assert` if everything is ready for modelling.

The purchase dates are added back to the `train` dataset using `merge()` on `ID`. We use this method instead of the previous notebooks `join()`, because that would a pandas PerformanceWarning related to DataFrame fragmentation. There is no difference as both methods serve the same purpose.

In [ ]:
train = pd.read_csv(DATA_DIR / "train_clean.csv")
test = pd.read_csv(DATA_DIR / "test_clean.csv")
sel_cols = pd.read_csv(DATA_DIR / "selected_columns.csv")
lists = pd.read_csv(DATA_DIR / "feature_lists.csv")
raw = pd.read_csv(DATA_DIR / "train_raw.csv")
sample_sub = pd.read_csv(DATA_DIR / "sample_submission.csv")

dates = raw[["ID", "purchase_date"]].set_index("ID")
train = (train.merge(raw[["ID", "purchase_date"]], on="ID", how="left").sort_values("purchase_date").reset_index(drop=True))

y = train[TARGET].to_numpy()
groups = train["customer_id"].to_numpy()

all_features = lists[lists["kind"].isin(["numeric_scaled", "indicator_flag", "one_hot"])]["feature"].tolist()
selected = sel_cols["column"].tolist()
group_lookup = dict(zip(sel_cols["column"], sel_cols["group"]))

print("Loaded:")
print(" train:", train.shape, "| test:", test.shape)
print(" rows ordered chronologically:", train["purchase_date"].min(), "->", train["purchase_date"].max())
print(" features selected by NB03:", len(selected), "| all NB02 features:", len(all_features))
print(" unique customers:", int(train["customer_id"].nunique()), "| max rows per customer:", int(train["customer_id"].value_counts().max()))
print("\nChecks:")
print(" selected set is a subset of all features:", all(c in set(all_features) for c in selected))
print(" no nulls in the modelling columns       :", bool(train[selected].isna().sum().sum() == 0))
print(" test IDs match the submission template  :", list(test["ID"]) == list(sample_sub["ID"]))
train_customers = set(train["customer_id"])
returning = [c in train_customers for c in test["customer_id"]]
given = [f for f in all_features + selected if f in ("ID", "customer_id")]
print(" identifiers reaching the model          :", given if given else "none")
assert not given, "the customer key must never become a model feature"
print(f" test rows whose customer is in train    : {int(sum(returning))} of {len(test)} = {100.0*sum(returning)/len(test):.1f}%")
assert all(c in set(all_features) for c in selected)
assert list(test["ID"]) == list(sample_sub["ID"]), "the submission must keep the template IDs in order"

### Cell 3 - Candidate feature sets:
Prepare two feature sets for model comparison: the selected features from Notebook 3 and all available features from Notebook 2, converting into NumPy arrays for `train` and `test`.

In [ ]:
feature_sets = {"NB03 selected": selected, "NB02 all features": all_features}
X_sets = {name: train[cols].to_numpy() for name, cols in feature_sets.items()}
X_test_sets = {name: test[cols].to_numpy() for name, cols in feature_sets.items()}

print("Candidate feature sets_")
for name, cols in feature_sets.items():
    print(f" {name:18s} {len(cols):4d} columns")
print("\n matrices come straight from train_clean.csv: no re-imputation, re-scaling or re-encoding")

### Cell 4 - Validation protocol
Compare three cross-validation methods using the same Logistic Regression model and the selected features from Notebook 3.

`StratifiedKFold`, `GroupKFold` and `TimeSeriesSplit` are compared using 5 folds and the F1 score, allowing us to compare their performance and variability. We also audit how each method splits the data in two directions: whether the same customer can appear in both training and validation, and whether a validation row is earlier than the newest row used to fit it. The second check is the one that decides the protocol here, because the whole `test` file is later than `train`.

In [ ]:
ref = LogisticRegression(max_iter=5000, random_state=RANDOM_STATE)
X_sel = X_sets["NB03 selected"]

protocols = {
    "StratifiedKFold (random)": StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE),
    "GroupKFold (customer_id)": GroupKFold(n_splits=5),
    "TimeSeriesSplit (chronological)": TimeSeriesSplit(n_splits=5),
}
prot_rows = []
for name, splitter in protocols.items():
    if name.startswith("GroupKFold"):
        res = cross_validate(ref, X_sel, y, cv=splitter, scoring=get_scorer("f1"), groups=groups)
    else:
        res = cross_validate(ref, X_sel, y, cv=splitter, scoring=get_scorer("f1"))
    prot_rows.append({"protocol": name, "cv_f1_mean": float(res["test_score"].mean()),
                      "cv_f1_std": float(res["test_score"].std()),
                      "min_fold": float(res["test_score"].min()),
                      "max_fold": float(res["test_score"].max())})
prot = pd.DataFrame(prot_rows)
print("=== Same model, three validation protocols ===")
print(prot.to_string(index=False, float_format=lambda v: "{:.4f}".format(v)))
print("\n  spread between protocols: {:.4f} F1".format(prot["cv_f1_mean"].max() - prot["cv_f1_mean"].min()))
print("  -> model selection uses TimeSeriesSplit: every test row is later than every train row,")
print("     so no fold may be fitted on data that comes after the rows it scores")
print("  -> the grouped number is kept as the never-seen-customer robustness check")
print("\n=== Audit: can a customer be on both sides of the split? ===")
purchase_dates = pd.to_datetime(train["purchase_date"]).to_numpy()
audit_rows = []
for name, splitter in protocols.items():
    span_customers, known_rows, fold_rows, past_rows = [], [], [], []
    fold_iter = splitter.split(X_sel, y, groups) if name.startswith("GroupKFold") else splitter.split(X_sel, y)
    for tr_idx, va_idx in fold_iter:
        in_fit = set(groups[tr_idx])
        is_returning = [c in in_fit for c in groups[va_idx]]
        span_customers.append(len([c for c in set(groups[va_idx]) if c in in_fit]))
        known_rows.append(int(sum(is_returning)))
        fold_rows.append(len(va_idx))
        past_rows.append(float(np.mean(purchase_dates[va_idx] < purchase_dates[tr_idx].max())))
    audit_rows.append({"protocol": name, "max_customers_on_both_sides": int(max(span_customers)),
                       "validation_rows_with_a_known_customer": int(sum(known_rows)),
                       "share_of_validation_rows": round(float(np.mean(
                           [a / b for a, b in zip(known_rows, fold_rows)])), 4),
                       "val_rows_earlier_than_last_train_row": round(float(np.mean(past_rows)), 4)})
audit = pd.DataFrame(audit_rows)


def audit_value(prefix, column):
    return audit[audit['protocol'].str.startswith(prefix)][column].tolist()[0]
display(audit.set_index('protocol'))
assert int(audit_value("GroupKFold", "max_customers_on_both_sides")) == 0, \
    "GroupKFold must keep every customer entirely inside one fold"
assert int(audit_value("GroupKFold", "validation_rows_with_a_known_customer")) == 0
assert float(audit_value("TimeSeriesSplit", "val_rows_earlier_than_last_train_row")) == 0.0, \
    "a chronological split must never score a row that is earlier than its own training rows"
print("\n  -> the last column is the one that decides this. GroupKFold is the only protocol that")
print("     keeps a customer on one side (0 shared rows), but {:.0f}% of its validation rows are".format(
    100 * float(audit_value("GroupKFold", "val_rows_earlier_than_last_train_row"))))
print("     EARLIER than the newest row it was fitted on, so it predicts the past out of the future.")
print("     TimeSeriesSplit is {:.0f}% on that measure, and it puts a returning customer on {:.0f}% of".format(
    100 * float(audit_value("TimeSeriesSplit", "val_rows_earlier_than_last_train_row")),
    100 * float(audit_value("TimeSeriesSplit", "share_of_validation_rows"))))
print("     its validation rows - legitimate there, because the fitting part is genuinely their past,")
print("     and it is the condition of the test file ({:.1f}% of its rows are returning customers).".format(
    100.0 * sum(returning) / len(test)))
print("     A random split carries both problems at once.")
print("  -> TimeSeriesSplit(5) therefore drives every decision below; GroupKFold stays in this")
print("     table as the never-seen-customer floor.")
STRICT = TimeSeriesSplit(n_splits=5)

The three validation methods give very close F1 scores, with `StratifiedKFold` the highest (0.7457), then `TimeSeriesSplit` (0.7410) and `GroupKFold` (0.7404). The fact that the three differ by 0.0053 suggests that the model's performance is relatively stable across all three strategies. The audit is what separates them. `GroupKFold` is the only protocol that never puts a customer on both sides, but 99.8% of its validation rows are earlier than the newest row it was fitted on, so it predicts the past out of the future. `TimeSeriesSplit` is 0% on that measure and keeps a returning customer on 86% of its validation rows, which is legitimate there because the fitting part is genuinely their past and it is the condition of the `test` file, 94.5% of whose rows are returning customers. In the customer analysis, with `StratifiedKFold` around 97% of validation rows belong to customers also present in training, so a random split carries both problems at once.

We choose `TimeSeriesSplit` for model selection, since no fold is ever fitted on rows that come after the rows it scores. `GroupKFold` is kept in the table as the floor for a customer we have never seen, which is the case for 5.5% of the `test` rows.

### Cell 5 - Scaling check and model selection:
Compare three scaling approaches: keeping the preprocessing from Notebook 2, re-scaling all features and re-scaling only numerical features.

We use the same Logistic Regression model, parameters and `TimeSeriesSplit` folds, evaluating each approach using F1 and ROC-AUC (Receiver Operating Characteristic: Area Under the Curve). In this case, ROC-AUT measures how well a model distinguishes between customers who will make another purchase and those who will not, across different classification thresholds.

We also compare four classification approaches: a Dummy Classifier (baseline), Logistic Regression, Decision Tree and PCA combined with Logistic Regression. Each model is tested with both feature sets from Notebooks 2 and 3. To find the best hyperparameters for each combination, `GridSearchCV` is with 5-fold `TimeSeriesSplit` is the method used, using F1 as the main evaluation metric. The results are organized into a table and a bar chart to compare candidates and model performance next stage.

In [ ]:
numeric_only = set(lists[lists["kind"] == "numeric_scaled"]["feature"])
def fold_scaled(X_fit, X_score, cols, mode):
    a, b = X_fit.copy(), X_score.copy()
    if mode == "nb02":
        return a, b
    keep = [i for i, c in enumerate(cols)
            if mode == "all" or c in numeric_only]
    sc = StandardScaler()
    a[:, keep] = sc.fit_transform(X_fit[:, keep])
    b[:, keep] = sc.transform(X_score[:, keep])
    return a, b
print("=== Scaling decision, same folds, same C, both metrics ===")
X_probe = X_sets["NB02 all features"]
probe_rows = []
for mode, label in [("nb02", "as Notebook 02 left it (0/1 columns stay 0/1)"),
                    ("all", "re-scale EVERY column inside the fold"),
                    ("numeric", "re-scale the numeric columns inside the fold")]:
    fold_f1, fold_auc = [], []
    for tr_idx, va_idx in STRICT.split(X_probe, y):
        a, b = fold_scaled(X_probe[tr_idx], X_probe[va_idx], all_features, mode)
        m = LogisticRegression(C=0.01, max_iter=5000, random_state=RANDOM_STATE).fit(a, y[tr_idx])
        prob = m.predict_proba(b)[:, 1]
        fold_f1.append(f1_score(y[va_idx], (prob >= 0.5).astype(int)))
        fold_auc.append(roc_auc_score(y[va_idx], prob))
    probe_rows.append({"scaling": label, "cv_f1_mean": float(np.mean(fold_f1)),
                       "roc_auc_mean": float(np.mean(fold_auc))})
    print(f"  {label:46s} CV F1 {np.mean(fold_f1):.4f}   ROC-AUC {np.mean(fold_auc):.4f}")
scaling_check = pd.DataFrame(probe_rows)
scaling_check.to_csv(MET_DIR / "scaling_check.csv", index=False)
print("\n  -> the 0/1 columns are left alone: re-scaling them costs ranking quality, and the")
print("     fold-level refit of the numeric columns changes nothing worth its complexity.")
print("     The models below therefore take the Notebook 02 matrix as it is.\n")

grids = {
    "dummy (most_frequent)": (Pipeline([("clf", DummyClassifier(strategy="most_frequent"))]), {}),
    "logistic": (Pipeline([("clf", LogisticRegression(max_iter=5000, random_state=RANDOM_STATE))]),
                 {"clf__C": [0.0001, 0.0003, 0.001, 0.003, 0.01, 0.03, 0.1, 0.3, 1.0, 3.0, 10.0],
                  "clf__class_weight": [None, "balanced"]}),
    "decision tree": (Pipeline([("clf", DecisionTreeClassifier(random_state=RANDOM_STATE))]),
                      {"clf__max_depth": [3, 5, 8, None],
                       "clf__min_samples_leaf": [1, 10, 25, 50, 100]}),
    "PCA + logistic": (Pipeline([("scale", StandardScaler()),
                                 ("pca", PCA(random_state=RANDOM_STATE)),
                                 ("clf", LogisticRegression(max_iter=5000, random_state=RANDOM_STATE))]),
                       {"pca__n_components": [5, 10, 20, 40],
                        "clf__C": [0.01, 0.1, 1.0, 10.0]}),
}

searches, rows = {}, []
for name, (pipe, grid) in grids.items():
    for set_name, X in X_sets.items():
        key = (name, set_name)
        search = GridSearchCV(pipe, grid, cv=STRICT, scoring=get_scorer("f1"), error_score=0)
        search.fit(X, y)
        searches[key] = search
        rows.append({"model": name, "feature_set": set_name,
                     "cv_f1_mean": float(search.best_score_),
                     "best_params": str(search.best_params_)})

full_df = pd.DataFrame(rows).sort_values("cv_f1_mean", ascending=False).reset_index(drop=True)
print("=== Every model x feature set, TimeSeriesSplit CV, scoring = F1 ===")
display(full_df.style.hide(axis="index"))

winner = full_df.to_dict("records")[0]
best_search = searches[(winner["model"], winner["feature_set"])]
base_estimator = best_search.best_estimator_
print(f"\n  selected : {winner['model']} on [{ winner['feature_set']}]")
print(f"  params   : {winner['best_params']}")
print(f"  CV F1    : {winner['cv_f1_mean']:.4f}")
print("  margin over the dummy baseline: {:+.4f} F1".format(
    winner["cv_f1_mean"] - float(full_df[full_df["model"].str.startswith("dummy")]["cv_f1_mean"].max())))

plot_df = full_df.copy()
short = {"NB02 all features": "all columns", "NB03 selected": "selected"}
plot_df["label"] = plot_df["model"] + " | " + plot_df["feature_set"].map(short)
plot_df = plot_df.sort_values("cv_f1_mean")
fig, ax = plt.subplots(figsize=(7.2, 2.7))
sns.barplot(data=plot_df, y="label", x="cv_f1_mean", color=PLOT_BLUE, ax=ax)
ax.axvline(float(plot_df[plot_df["model"].str.startswith("dummy")]["cv_f1_mean"].max()), color=PLOT_ORANGE, linestyle="--", label="dummy baseline")
ax.set_title("Models compared under the strict protocol (TimeSeriesSplit, chronological)")
ax.set_xlabel("TimeSeriesSplit CV F1")
ax.set_ylabel("")
ax.set_xlim(min(0.70, float(plot_df["cv_f1_mean"].min()) - 0.01), float(plot_df["cv_f1_mean"].max()) + 0.012)
ax.legend(loc="upper right", fontsize=8)
plt.setp(ax.get_yticklabels(), fontsize=8)
guide(ax, "How to read: longer bar = better chronological F1; dashed = baseline.")
plt.tight_layout()
plt.savefig(FIG_DIR / "nb04_model_comparison.png", dpi=200)
plt.show()

The scaling comparison shows very similar results across the three methods. Keeping the preprocessing from Notebook 2 achieved the highest F1 score (0.7606), re-scaling only numerical features was within 0.001 of it (0.7599) and matched its ROC-AUC (0.7948), while re-scaling every column was clearly worse (0.7494). Since the differences between the first two are very small, we decided to keep the original preprocessing from Notebook 2.

In the model comparison, the winner was **Logistic Regression using all features from Notebook 2** (F1=0.7606, against 0.7576 for the same model on the Notebook 3 selected set and 0.7501 for the best PCA+Logistic), with hyperparameter `C = 0.01` and no class weighting. This represents an improvement of 0.0398 over the Dummy Classifier baseline. So, Logistic Regression is a suitable choice for this machine learning model. Keeping all features provides better performance than the other combinations tested, although under this protocol the margin over the selected set is narrow.

### Cell 6 - Classification threshold optimization:
Adjust the classification threshold to improve its F1 score.

In this case, by default Logistic Regression uses a threshold of 0.5 to convert probabilities into class predictions, but this value may not be the best choice for our model. To find a better threshold, we apply nested cross-validation with `TimeSeriesSplit`. For each of the 5 outer folds, we use a 4-fold chronological cross-validation inside its training part to find the threshold that gives us the highest F1 score. Then, the selected threshold is tested on the outer validation fold, using data that was not involved in its selection.

In [ ]:
def f1_curve(target, probs):
    precision, recall, thresholds = precision_recall_curve(target, probs)
    with np.errstate(divide="ignore", invalid="ignore"):
        f1s = np.where(precision + recall > 0, 2 * precision * recall / (precision + recall), 0.0)
    return precision, recall, thresholds, f1s[:-1]


def best_threshold(probs, target):
    _, _, thresholds, f1s = f1_curve(target, probs)
    if len(thresholds) == 0:
        return 0.5, 0.0
    k = int(np.argmax(f1s))
    return float(thresholds[k]), float(f1s[k])
X_best = X_sets[winner["feature_set"]]
nested_rows = []
for fold, (tr_idx, va_idx) in enumerate(TimeSeriesSplit(n_splits=5).split(X_best, y), start=1):
    inner_X, inner_y = X_best[tr_idx], y[tr_idx]
    inner_oof = np.full(len(tr_idx), np.nan)
    for itr, iva in TimeSeriesSplit(n_splits=4).split(inner_X, inner_y):
        m = clone(base_estimator)
        m.fit(inner_X[itr], inner_y[itr])
        inner_oof[iva] = m.predict_proba(inner_X[iva])[:, 1]
    inner_covered = ~np.isnan(inner_oof)
    thr, inner_f1 = best_threshold(inner_oof[inner_covered], inner_y[inner_covered])

    outer_model = clone(base_estimator)
    outer_model.fit(X_best[tr_idx], y[tr_idx])
    outer_probs = outer_model.predict_proba(X_best[va_idx])[:, 1]
    nested_rows.append({"fold": fold, "inner_threshold": thr, "inner_f1": inner_f1,
                        "f1_at_tuned": f1_score(y[va_idx], (outer_probs >= thr).astype(int)),
                        "f1_at_0_50": f1_score(y[va_idx], (outer_probs >= 0.5).astype(int))})

nested = pd.DataFrame(nested_rows)
print("=== Nested threshold tuning (inner chooses, outer evaluates) ===")
display(nested.set_index("fold"))
print("\n  mean outer F1 at the tuned threshold : {:.4f} (± {:.4f})".format(
    nested["f1_at_tuned"].mean(), nested["f1_at_tuned"].std()))
print("  mean outer F1 at the default 0.50      : {:.4f} (± {:.4f})".format(
    nested["f1_at_0_50"].mean(), nested["f1_at_0_50"].std()))
print("  measured gain from tuning the threshold: {:+.4f} F1".format(
    nested["f1_at_tuned"].mean() - nested["f1_at_0_50"].mean()))

The nested chronological cross-validation shows that under the chronological protocol, tuning the threshold doesn't reveal an improvement on unseen validation folds: the model's mean F1 score moves from **0.7606 at the default 0.5 to 0.7587 at the tuned threshold**, a change of -0.0019, and the standard deviation rises from **0.0234 to 0.0245**. The candidate thresholds in the inner folds also spread from 0.4164 to 0.5082, a considerable range, because each inner fold is fitted on a different period. Therefore, this nested evaluation does not provide evidence that threshold optimization in general reveals an improvement. However, its purpose is to assess the threshold tuning procedure rather than select a single threshold for the final model. A final threshold will be selected separately in Cell 7 using the available outer fold predictions.

### Cell 7 - Final model:
This cell trains the selected model using the `train` dataset and the best hyperparameters found earlier.

Unlike Cell 6, which used nested chronological cross-validation to evaluate the threshold optimization procedure, here a 5-fold `TimeSeriesSplit` is used to generate out-of-fold (oof) predictions and select a final classification threshold based on the F1 score, comparing it with the default threshold of 0.5. A chronological split cannot score its own earliest rows, so 5390 of the 6469 observations receive an out-of-fold prediction and the first 1079 are training only; every out-of-fold metric below is computed on those covered rows. We perform a chronological holdout, with training on the earliest 80% of observations and validation on the latest 20%. This allows us to compare the two thresholds on more recent purchases and check if the proportion of the target changes over time.

In [ ]:
final_model = clone(base_estimator)
final_model.fit(X_best, y)

oof = np.full(len(y), np.nan)
for tr_idx, va_idx in TimeSeriesSplit(n_splits=5).split(X_best, y):
    m = clone(base_estimator)
    m.fit(X_best[tr_idx], y[tr_idx])
    oof[va_idx] = m.predict_proba(X_best[va_idx])[:, 1]

covered = ~np.isnan(oof)
y_oof, oof_oof = y[covered], oof[covered]
THRESHOLD, thr_f1 = best_threshold(oof_oof, y_oof)
print("=== Final model ===")
print(" model      :", winner["model"], "on", winner["feature_set"])
print(" params     :", winner["best_params"])
print(" TimeSeriesSplit CV F1 : {:.4f}".format(winner["cv_f1_mean"]))
print("\n=== Out-of-fold coverage ===")
print("  a chronological split cannot score its own earliest rows: {} of {} rows receive an".format(int(covered.sum()), len(y)))
print("  out-of-fold prediction, and the first {} are training-only, so every out-of-fold".format(int((~covered).sum())))
print("  metric below is computed on the covered rows alone.")
print("\n=== Threshold ===")
print(f" chosen for F1 on out-of-fold probabilities : {THRESHOLD:.4f}")
print(f" out-of-fold F1 at that threshold           : {thr_f1:.4f}")
print(f" out-of-fold F1 at the default 0.50         : {f1_score(y_oof, (oof_oof >= 0.5).astype(int)):.4f}")

cut = int(0.8 * len(y))
m_ch = clone(base_estimator)
m_ch.fit(X_best[:cut], y[:cut])
p_ch = m_ch.predict_proba(X_best[cut:])[:, 1]
print("\n=== Chronological holdout (fit earliest 80%, score latest 20%) ===")
print("  F1 at the default 0.50     : {:.4f}".format(f1_score(y[cut:], (p_ch >= 0.5).astype(int))))
print("  F1 at the chosen threshold : {:.4f}   (the threshold was picked on the chronological folds, not here)".format(
    f1_score(y[cut:], (p_ch >= THRESHOLD).astype(int))))
print("  positive rate in the held-out tail : {:.4f}  vs {:.4f} in the earlier rows".format(y[cut:].mean(), y[:cut].mean()))

clf = final_model.named_steps["clf"]
readout_name, readout_col = ("coefficient", "coef") if hasattr(clf, "coef_") else ("importance", "importance")
values = clf.coef_[0] if readout_name == "coefficient" else clf.feature_importances_
readout = pd.DataFrame({"column": feature_sets[winner["feature_set"]],
                        readout_col: np.asarray(values).ravel()})
readout["group"] = [group_lookup.get(c, c) for c in readout["column"]]
readout["abs"] = readout[readout_col].abs()
readout = readout.sort_values("abs", ascending=False).reset_index(drop=True)
print(f"\n=== Model read-out: {readout_name} of every column, largest effect first ===")
display(readout.set_index("column"))

The final model is **Logistic Regression with all features from Notebook 2**, using `C = 0.01` and no class weighting. It achieved a `TimeSeriesSplit` F1 score of **0.7606**. The selected threshold was **0.4651**, using outer fold results, moving the F1 score from 0.7618 (default threshold of 0.5) to **0.7658** on the 5390 covered rows; the nested check above puts that same gain at -0.0019, so it should be read as within noise rather than as an improvement. The chronological holdout process also moved the F1 score from **0.7766 to 0.7854**. Looking at the model coefficients, purchase history features, such as `purchase_count_365d` and `purchase_count_90d`, have some of the largest coefficients, along with certain purchase months encoded features, in other words, they have a greater influence on the predictions of the logistic regression model.

### Cell 8 - Assessment:
Calculate the final model precision, recall, F1 score, accuracy and ROC-AUC and display of confusion matrix (in out-of-fold results, on the 5390 rows a chronological split can score) and plots to understand the model performance and how different thresholds affect the results.

We also examine the chronological holdout in more detail by separating customers into two groups: those who have appeared in the `train` dataset and those making their first purchase. We compare their F1 and ROC-AUC scores to understand how the model performs for different customer groups.

In [ ]:
oof_preds = (oof_oof >= THRESHOLD).astype(int)
cm = confusion_matrix(y_oof, oof_preds)
tn, fp, fn, tp = cm.ravel()

print("=== Out-of-fold assessment at the tuned threshold ({} of {} rows covered) ===".format(len(y_oof), len(y)))
print(classification_report(y_oof, oof_preds, target_names=["no repeat (0)", "repeat (1)"]))
print(f"  confusion matrix : TN={tn}  FP={fp}  FN={fn}  TP={tp}")
print(f"  ROC-AUC (threshold-free) : {roc_auc_score(y_oof, oof_oof):.4f}")
print("  accuracy                 : {:.4f}   (always-majority baseline = {:.4f})".format(accuracy_score(y_oof, oof_preds), max(y_oof.mean(), 1 - y_oof.mean())))

precision, recall, thresholds, f1s = f1_curve(y_oof, oof_oof)

fig, axes = plt.subplots(2, 2, figsize=(7.2, 4.7))
sns.heatmap(pd.DataFrame(cm, index=["true 0", "true 1"], columns=["pred 0", "pred 1"]),
            annot=True, fmt="d", cmap="Blues", ax=axes[0, 0])
axes[0, 0].set_title("Confusion matrix (out-of-fold)")
axes[0, 0].set_xlabel("predicted")
axes[0, 0].set_ylabel("actual")

axes[0, 1].plot(recall, precision, color=PLOT_BLUE, label="precision")
axes[0, 1].plot(recall, 1 - recall, color="black", linestyle=":", label="1 - recall")
axes[0, 1].plot(recall[:-1], f1s, color=PLOT_ORANGE, linestyle="--", label="F1")
axes[0, 1].set_xlabel("Recall")
axes[0, 1].set_title("Precision and F1 on the PR curve")
axes[0, 1].legend()

axes[1, 0].plot(thresholds, f1s, color=PLOT_BLUE)
axes[1, 0].axvline(THRESHOLD, color=PLOT_ORANGE, linestyle="--", label="chosen {:.3f}".format(THRESHOLD))
axes[1, 0].axvline(0.5, color="black", linestyle=":", label="default 0.50")
axes[1, 0].set_xlabel("Decision threshold")
axes[1, 0].set_ylabel("F1")
axes[1, 0].set_title("F1 vs threshold (out-of-fold)")
axes[1, 0].legend()

fpr, tpr, _ = roc_curve(y_oof, oof_oof)
axes[1, 1].plot(fpr, tpr, color=PLOT_BLUE, label="AUC {:.4f}".format(roc_auc_score(y_oof, oof_oof)))
axes[1, 1].plot([0, 1], [0, 1], color="black", linestyle="--", label="random")
axes[1, 1].set_xlabel("False positive rate")
axes[1, 1].set_ylabel("True positive rate")
axes[1, 1].set_title("ROC curve (out-of-fold)")
axes[1, 1].legend()
guide(axes[0, 0], "How to read: rows actual, cols predicted.")
guide(axes[0, 1], "How to read: right = more recall.")
guide(axes[1, 0], "How to read: orange = chosen, dotted = 0.50.")
guide(axes[1, 1], "How to read: AUC in the legend is threshold-free.")
plt.tight_layout()
plt.savefig(FIG_DIR / "nb04_assessment.png", dpi=200)
plt.show()

cut = int(0.8 * len(y))
chrono_model = clone(base_estimator)
chrono_model.fit(X_best[:cut], y[:cut])
chrono_prob = chrono_model.predict_proba(X_best[cut:])[:, 1]
fit_customers = set(groups[:cut])
hold = [i for i in range(cut, len(y))]
returning_in_holdout = [i for i in hold if groups[i] in fit_customers]
new_in_holdout = [i for i in hold if groups[i] not in fit_customers]

holdout_rows = []
for label, idx in [("returning customer (seen before the cut)", returning_in_holdout),
                   ("new customer (first appearance)", new_in_holdout),
                   ("all holdout rows", hold)]:
    if not idx:
        continue
    t = y[idx]
    p = chrono_prob[[k - cut for k in idx]]
    pos_rate = float(np.mean(t))
    holdout_rows.append({"group": label, "rows": len(idx),
                         "share_of_holdout": round(len(idx) / len(hold), 4),
                         "positive_rate": round(pos_rate, 4),
                         "always_yes_f1": round(2 * pos_rate / (1 + pos_rate), 4),
                         "f1_at_0_50": round(f1_score(t, (p >= 0.5).astype(int)), 4),
                         "f1_at_chosen_threshold": round(f1_score(t, (p >= THRESHOLD).astype(int)), 4),
                         "roc_auc": round(roc_auc_score(t, p), 4)})
holdout_table = pd.DataFrame(holdout_rows)
print(f"\n=== Chronological holdout ({len(hold)} of {len(y)} rows), split by whether the customer was already seen ===")
print("  always_yes_f1 = F1 of predicting repeat for every row = 2 * positive_rate / (1 + positive_rate)")
print(holdout_table.to_string(index=False))
holdout_table.to_csv(MET_DIR / "holdout_by_customer_type.csv", index=False)
print(f"\n  -> returning customers are scored better, and {100.0*sum(returning)/len(test):.0f}% of the test dataset is made of them,")
print("     which is why the chronological protocol carries the headline and the grouped number")
print("     is quoted as a floor for the never-seen-customer case.")

The final model achieved an **F1 score=0.77** on the 5390 covered rows, with a precision of 0.73 and a recall of 0.80 for the repeat class. It predicts a false negative rate of 19.5% (593 out of 3041 customers who repeat purchases) and a false positive rate of 38.5% (904 out of 2349 customers who don't repeat purchases). It can therefore be said that the model is more likely to predict an actual outcome 0 as 1 than an actual outcome 1 as 0. The **ROC-AUC=0.7939** indicates a reasonable ability to distinguish between the two classes. The accuracy of **0.7223** is also higher than the majority-class baseline of 0.5642 on the same rows.

In the chronological holdout, the model performed better for returning customers (**F1=0.7926**) than for new customers (**F1=0.7101**), using the selected threshold. It has also been proven that the threshold improved the F1 score for both groups compared with the default value of 0.5.

### Cell 9 - `Test` predictions
Predict the `test` set and creates the outcome file.

We also did a threshold sensitivity analysis, to check if it affects the predicted positive rate in the `test` dataset, comparing different threshold values.

In [ ]:
test_probs = final_model.predict_proba(X_test_sets[winner["feature_set"]])[:, 1]
test_preds = (test_probs >= THRESHOLD).astype(int)

submission = pd.DataFrame({"ID": test["ID"].to_numpy(), TARGET: test_preds})
submission.to_csv(DELIV_DIR / SUBMISSION_FILE, index=False)
pd.DataFrame({"ID": test["ID"].to_numpy(), "prob_repeat_90d": test_probs, "pred_at_chosen_threshold": test_preds,
              "pred_at_0_50": (test_probs >= 0.5).astype(int)}).to_csv(MET_DIR / "test_predictions.csv", index=False)

print("=== Submission contract check ===")
print("  file                                : deliverables/{}".format(SUBMISSION_FILE))
print("  columns == sample_submission        :", list(submission.columns) == list(sample_sub.columns))
print("  rows                                : {} (template {})".format(len(submission), len(sample_sub)))
print("  IDs identical to template, in order :", list(submission["ID"]) == list(sample_sub["ID"]))
print("  predictions are 0/1                 :", sorted(submission[TARGET].unique().tolist()) == [0, 1])
print("  nulls                               :", int(submission.isna().sum().sum()))
print("  predicted positive rate             : {:.4f}  (train {:.4f})".format(
    submission[TARGET].mean(), y.mean()))
print("  test probability mean / median      : {:.4f} / {:.4f}".format(
    float(test_probs.mean()), float(np.median(test_probs))))
print("  positive rate at 0.50               : {:.4f}".format(float((test_probs >= 0.5).mean())))
print("  how many predictions the threshold moved vs 0.50: {}".format(
    int((submission[TARGET] != (test_probs >= 0.5).astype(int)).sum())))
print("  A higher positive rate than train is expected: the test rows are later and their")
print("  customers are longer-tenured (NB01/NB02), and the held-out chronological tail above")
print("  already shows a higher positive rate than the earlier rows.")
sens = pd.DataFrame([{"threshold": t, "oof_f1_chronological": f1_score(y_oof, (oof_oof >= t).astype(int)), "f1_chronological_holdout": f1_score(y[cut:], (p_ch >= t).astype(int)),
                      "implied_test_positive_rate": float((test_probs >= t).mean())} for t in [0.35, 0.40, round(float(THRESHOLD), 4), 0.45, 0.50]]).sort_values("threshold").reset_index(drop=True)
print("\n=== Threshold sensitivity ===")
print(sens.to_string(index=False, float_format=lambda v: "{:.4f}".format(v)))
chosen_row = sens.loc[sens["threshold"] == round(float(THRESHOLD), 4)].iloc[0]
default_row = sens.loc[sens["threshold"] == 0.50].iloc[0]
plateau = sens.loc[sens["oof_f1_chronological"] >= chosen_row["oof_f1_chronological"] - 0.002, "threshold"]
print("  the plateau around the chosen threshold runs from {:.2f} to {:.2f}, so the choice is not".format(
    float(plateau.min()), float(plateau.max())))
print("  knife-edge; moving to the default 0.50 costs {:+.4f} F1 on the chronological folds and".format(
    float(default_row["oof_f1_chronological"] - chosen_row["oof_f1_chronological"])))
print("  {:+.4f} on the held-out tail, which is why it is not used.".format(
    float(default_row["f1_chronological_holdout"] - chosen_row["f1_chronological_holdout"])))
sens.to_csv(MET_DIR / "threshold_sensitivity.csv", index=False)

assert list(submission.columns) == list(sample_sub.columns)
assert list(submission["ID"]) == list(sample_sub["ID"])
assert int(submission.isna().sum().sum()) == 0
print("\n  first rows:")
print(submission.set_index('ID').head())

The final predictions of `test` dataset were successfully generated and the submission file passed all validation checks (`assert` statements). It contains 986 rows, with the correct columns, binary target and no missing values. Using the selected threshold of 0.4651, the model predicts that 70.69% observations will have customers repeating purchase within 90 days, compared with 66.33% using the default threshold of 0.5. As the threshold sensitivity analysis explains, values between approximately 0.45 and 0.47 produce similar F1 scores.

### Cell 10 - Export results and final summary:
Save main results from Notebook 4, including the selected model, its hyperparameters, classification threshold and performance statistics.

All relevant information were stored in JSON and CSV files.

In [ ]:
def to_native(v):
    if isinstance(v, (np.integer,)):
        return int(v)
    if isinstance(v, (np.floating,)):
        return float(v)
    if v is None or isinstance(v, (bool, int, float, str)):
        return v
    return str(v)

metrics = {
    "model": winner["model"],
    "feature_set": winner["feature_set"],
    "n_features": len(feature_sets[winner["feature_set"]]),
    "best_params": winner["best_params"],
    "winner_model_cv_f1_chronological": to_native(winner["cv_f1_mean"]),
    "oof_rows_covered": int(covered.sum()),
    "oof_rows_total": int(len(y)),
    "oof_f1_at_chosen_threshold": to_native(thr_f1),
    "oof_f1_at_0_50": to_native(f1_score(y_oof, (oof_oof >= 0.5).astype(int))),
    "nested_honest_f1": to_native(nested["f1_at_tuned"].mean()),
    "nested_honest_f1_std": to_native(nested["f1_at_tuned"].std()),
    "chosen_threshold": to_native(THRESHOLD),
    "roc_auc": to_native(roc_auc_score(y_oof, oof_oof)),
    "accuracy": to_native(accuracy_score(y_oof, oof_preds)),
    "always_majority_f1": to_native(2 * y_oof.mean() / (1 + y_oof.mean())),
    "test_positive_rate": to_native(test_preds.mean()),
    "protocol_reference_cv_f1_chronological": to_native(prot[prot["protocol"].str.startswith("TimeSeries")]["cv_f1_mean"].tolist()[0]),
    "chronological_holdout_f1": to_native(f1_score(y[cut:], (chrono_prob >= THRESHOLD).astype(int))),
    "holdout_f1_returning_customer": to_native(
        holdout_table[holdout_table["group"].str.startswith("returning")]["f1_at_chosen_threshold"].tolist()[0]),
    "holdout_f1_new_customer": to_native(
        holdout_table[holdout_table["group"].str.startswith("new")]["f1_at_chosen_threshold"].tolist()[0]),
    "test_rows_from_returning_customers": to_native(sum(returning) / len(test)),
    "grouped_val_rows_with_a_known_customer": to_native(float(audit_value("GroupKFold", "share_of_validation_rows"))),
    "random_split_val_rows_with_a_known_customer": to_native(float(audit_value("StratifiedKFold", "share_of_validation_rows"))),
}
with open(MET_DIR / "assessment.json", "w", encoding="utf-8") as fh:
    json.dump(metrics, fh, indent=2)

prot.to_csv(MET_DIR / "validation_protocols.csv", index=False)
full_df.to_csv(MET_DIR / "model_comparison.csv", index=False)
nested.to_csv(MET_DIR / "threshold_nested_cv.csv", index=False)
readout.to_csv(MET_DIR / "model_readout.csv", index=False)
pd.DataFrame({"threshold": thresholds, "f1": f1s}).to_csv(MET_DIR / "f1_by_threshold.csv", index=False)

pd.DataFrame({"threshold": [THRESHOLD], "tn": [int(tn)], "fp": [int(fp)], "fn": [int(fn)],
              "tp": [int(tp)]}).to_csv(MET_DIR / "confusion_matrix.csv", index=False)
pd.DataFrame({"ID": train["ID"], "customer_id": train["customer_id"], "y_true": y,
              "oof_prob": oof}).to_csv(MET_DIR / "oof_predictions.csv", index=False)

print("=== Notebook 04 summary ===")
for k, v in metrics.items():
    print("  {:32s}: {}".format(k, v))
print("\n  Estimated F1, nested chronological cross-validation                  : {:.4f}".format(metrics["nested_honest_f1"]))
print("  Estimated F1, reference model, chronological CV at the default 0.50  : {:.4f}".format(metrics["protocol_reference_cv_f1_chronological"]))
print("  Chronological holdout at the submitted threshold                     : {:.4f}".format(metrics["chronological_holdout_f1"]))
print("\n  Every number above comes from TimeSeriesSplit, so no fold is ever fitted on rows that")
print("  come after the rows it scores. GroupKFold stays in the protocol table as the floor for")
print("  a customer we have never seen; {:.0f}% of the test rows are not that case, and the holdout".format(100.0 * metrics["test_rows_from_returning_customers"]))
print("  scores returning customers at {:.4f} against {:.4f} for new ones.".format(metrics["holdout_f1_returning_customer"], metrics["holdout_f1_new_customer"]))
print("  Chronological validation also matches the test dataset, whose observations are later")
print("  than those in training and whose customers are longer-tenured (NB01/NB02).")

---
## Where this notebook leaves the project

**Produced files**: `deliverables/ML_Group_22_Version05.csv`, every metric in `artifacts/metrics/` (including `test_predictions.csv` with the raw probabilities) and the assessment figure in `artifacts/figures/`.


**Final deliverable**: overall structure of our pipeline in PDF, `Group_22_handout_text.pdf`.

**Note:** Reproducing everything from a clean clone, by creating the environment from `environment.yml` or `requirements.txt`, put the four supplied CSVs in `data/`, then run the notebooks in order. No folder is created by code in the notebooks.

---
---


**Group 22 members:**

Mohamed Triki  
Wael Farhat  
Gustavo Carvalho